Installing Packages

In [16]:
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
try:
    install("openai-whisper")
    install("sentence-transformers")
    install("sumy")
    install("python-louvain")
    install("google-generativeai")
    install("gradio")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"])

    print("Dependencies installed!\n")
except Exception as e:
    print(f"nstallation warning: {e}")
    print("Continuing with available packages...\n")

Dependencies installed!

Dependencies installed!



Imports

In [17]:
import warnings
warnings.filterwarnings('ignore')

import os, re, nltk, torch, traceback, time
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
from collections import defaultdict, Counter
try:
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.cluster import AgglomerativeClustering
    SKLEARN_AVAILABLE = True
except ImportError:
    print("sklearn unavailable, using numpy alternatives")
    SKLEARN_AVAILABLE = False
    def cosine_similarity(X, Y=None):
        if Y is None:
            Y = X
        X = np.array(X)
        Y = np.array(Y)
        X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
        Y_norm = Y / np.linalg.norm(Y, axis=1, keepdims=True)
        return np.dot(X_norm, Y_norm.T)
    class AgglomerativeClustering:
        def __init__(self, n_clusters=2, **kwargs):
            self.n_clusters = n_clusters

        def fit_predict(self, X):
            n_samples = len(X)
            if n_samples <= self.n_clusters:
                return np.arange(n_samples)
            np.random.seed(42)
            centers_idx = np.random.choice(n_samples, self.n_clusters, replace=False)

            for _ in range(10):
                similarities = cosine_similarity(X, X[centers_idx])
                labels = np.argmax(similarities, axis=1)
                for i in range(self.n_clusters):
                    mask = labels == i
                    if mask.any():
                        centers_idx[i] = np.where(mask)[0][0]

            return labels

from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import community.community_louvain as community_louvain

import whisper
import google.generativeai as genai
import gradio as gr
import spacy

print("All imports completed\n")



All imports completed

All imports completed



NLTK Setup

In [18]:
for resource in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource, quiet=True)
print("NLTK ready\n")
try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy loaded\n")
except:
    print("spaCy model not loaded (optional)\n")
    nlp = None



NLTK ready

spaCy loaded

NLTK ready

spaCy loaded



Configuration

In [19]:
class MDSSConfig:
    WHISPER_MODEL = "tiny"  #switch models for better performance
    EMBEDDING_MODEL = "all-MiniLM-L6-v2"
    SARCASM_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    MIN_MODULE_SIZE = 2
    EXTRACTIVE_SENTENCES = 3
    REDUNDANCY_THRESHOLD = 0.85
    AUTO_DETECT_SPEAKERS = True
    MAX_SPEAKERS = 6
    API_DELAY = 7
    RETRY_ATTEMPTS = 3
    RETRY_DELAY = 10

    # Batching
    BATCH_SIZE_INTENTS = 999
    BATCH_SIZE_MODULES = 2

config = MDSSConfig()



Rate Limiter

In [20]:
class RateLimiter:
    def __init__(self, delay=7):
        self.delay = delay
        self.last_call = 0

    def wait(self):
        """Wait if needed to respect rate limits"""
        elapsed = time.time() - self.last_call
        if elapsed < self.delay:
            wait_time = self.delay - elapsed
            print(f"Rate limit: waiting {wait_time:.1f}s...")
            time.sleep(wait_time)
        self.last_call = time.time()

!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content("Say ready")
print(response.text)



Ready!
Ready!


Gemini setup

In [21]:
def setup_gemini_api():
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY not found")

        genai.configure(api_key=api_key)
        try:
            available_models = [m.name for m in genai.list_models()
                              if 'generateContent' in m.supported_generation_methods]
            if not available_models:
                raise RuntimeError("No Gemini models available")
            model_name = None
            for prefix in ['models/gemini-2.5-flash', 'models/gemini-1.5-flash', 'models/gemini']:
                for m in available_models:
                    if m.startswith(prefix):
                        model_name = m
                        break
                if model_name:
                    break

            if not model_name:
                model_name = available_models[0]

        except Exception as e:
            print(f"Model listing failed: {e}")
            model_name = "models/gemini-1.5-flash"
        model = genai.GenerativeModel(model_name)
        response = model.generate_content("Say 'ready'")
        print(f"Gemini connected: {model_name}")
        print(f"Test: {response.text.strip()}\n")
        return True, model_name

    except Exception as e:
        print(f"\nGEMINI SETUP FAILED: {e}")
        print("\nTo fix:")
        print("   1. Click the key in left sidebar")
        print("   2. Add: GEMINI_API_KEY")
        print("   3. Get key: https://aistudio.google.com/apikey\n")
        return False, None

GEMINI_READY, GEMINI_MODEL = setup_gemini_api()
if not GEMINI_READY:
    raise RuntimeError("Set up Gemini API key first!")



Gemini connected: models/gemini-2.5-flash
Test: Ready

Gemini connected: models/gemini-2.5-flash
Test: Ready.



Speaker Diarization

In [22]:
class SpeakerDiarization:
    def __init__(self):
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)
        try:
            self.nlp = spacy.load("en_core_web_sm")
            self.spacy_available = True
        except:
            print("spaCy unavailable")
            self.nlp = None
            self.spacy_available = False

        print("Ready\n")

    def _extract_speaker_names(self, utterances):
        """Extract speaker names from dialogue text (e.g., 'Name: text' or 'A (role): text')"""
        name_to_utterances = {}

        for idx, utt in enumerate(utterances):
            match = re.match(r'^([A-Z])\s*(?:\([^)]+\))?\s*[:\-]\s*(.+)', utt)
            if match:
                name = match.group(1).strip()
                if name not in name_to_utterances:
                    name_to_utterances[name] = []
                name_to_utterances[name].append(idx)
                continue
            match = re.match(r'^([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s*[:\-]\s*(.+)', utt)
            if match:
                name = match.group(1).strip()
                if name not in name_to_utterances:
                    name_to_utterances[name] = []
                name_to_utterances[name].append(idx)
        covered = sum(len(indices) for indices in name_to_utterances.values())
        if covered >= len(utterances) * 0.3 and len(name_to_utterances) >= 2:
            return name_to_utterances

        return None

    def _estimate_speakers(self, embeddings, max_speakers=5, min_score=0.15):
        from sklearn.metrics import silhouette_score, calinski_harabasz_score

        if len(embeddings) < 3:
            return min(2, len(embeddings))

        best_score = -1
        best_k = 2

        for k in range(2, min(max_speakers + 1, len(embeddings))):
            try:
                clustering = AgglomerativeClustering(n_clusters=k, linkage='ward')
                labels = clustering.fit_predict(embeddings)

                sil_score = silhouette_score(embeddings, labels)
                ch_score = calinski_harabasz_score(embeddings, labels)

                label_counts = Counter(labels)
                balance_penalty = np.std(list(label_counts.values())) / (np.mean(list(label_counts.values())) + 1e-8)

                combined_score = sil_score * (ch_score / 1000) * (1 / (1 + balance_penalty * 0.5))

                if combined_score > best_score:
                    best_score = combined_score
                    best_k = k
            except:
                continue
        if best_score < min_score * 0.5:
            print(f"Low confidence ({best_score:.3f}), defaulting to 2 speakers")
            return 2

        return best_k

    def _extract_style_features(self, utterances):
        features = []

        for utt in utterances:
            words = utt.split()
            feat = {
                'length': len(words),
                'has_question': 1 if '?' in utt else 0,
                'exclamation': utt.count('!'),
                'uppercase_ratio': sum(1 for c in utt if c.isupper()) / (len(utt) + 1),
                'avg_word_len': np.mean([len(w) for w in words]) if words else 0
            }
            features.append(list(feat.values()))

        return np.array(features)

    def _refine_with_turn_taking(self, embeddings, labels):
        refined_labels = labels.copy()
        n_samples = len(embeddings)
        for i in range(1, n_samples - 1):
            if refined_labels[i] == refined_labels[i-1] == refined_labels[i+1]:
                same_speaker_sim = np.dot(embeddings[i], embeddings[i-1])
                other_speakers = [j for j in range(n_samples)
                                if refined_labels[j] != refined_labels[i]]

                if other_speakers:
                    sample_speakers = other_speakers[:min(5, len(other_speakers))]
                    other_sims = [np.dot(embeddings[i], embeddings[j])
                                for j in sample_speakers]
                    max_other_sim = max(other_sims)
                    if max_other_sim > same_speaker_sim * 1.15:
                        for j in sample_speakers:
                            if np.dot(embeddings[i], embeddings[j]) == max_other_sim:
                                refined_labels[i] = refined_labels[j]
                                break
        return refined_labels
    def _calculate_confidences(self, embeddings, labels):
        confidences = []

        for i, label in enumerate(labels):
            cluster_mask = labels == label
            cluster_embeddings = embeddings[cluster_mask]
            cluster_center = cluster_embeddings.mean(axis=0)
            dist_to_center = np.linalg.norm(embeddings[i] - cluster_center)
            other_labels = [l for l in set(labels) if l != label]
            if other_labels:
                min_other_dist = float('inf')
                for other_label in other_labels:
                    other_mask = labels == other_label
                    other_center = embeddings[other_mask].mean(axis=0)
                    other_dist = np.linalg.norm(embeddings[i] - other_center)
                    min_other_dist = min(min_other_dist, other_dist)
                confidence = min(0.99, min_other_dist / (dist_to_center + min_other_dist + 1e-8))
            else:
                confidence = 0.95
            confidences.append(float(confidence))
        return confidences
    def diarize(self, utterances, num_speakers=None, use_names=True,
                use_turn_taking=True, use_style_features=True):
        """
        Advanced speaker diarization with multiple strategies

        Args:
            utterances: List of text utterances
            num_speakers: Fixed number (None for auto-detect)
            use_names: Try extracting speaker names from text
            use_turn_taking: Refine using conversation flow patterns
            use_style_features: Include linguistic style features
        """
        if not utterances or len(utterances) < 2:
            return [{"speaker": "SPEAKER_0", "text": u, "confidence": 1.0}
                   for u in utterances]
        if use_names:
            name_mapping = self._extract_speaker_names(utterances)

            if name_mapping:
                print(f"Extracted {len(name_mapping)} speaker names: {list(name_mapping.keys())}")
                diarized = []
                speaker_names = list(name_mapping.keys())

                for idx, utt in enumerate(utterances):
                    speaker = None
                    for name, indices in name_mapping.items():
                        if idx in indices:
                            speaker = name
                            break
                    if speaker is None:
                        min_dist = float('inf')
                        for name, indices in name_mapping.items():
                            for i in indices:
                                dist = abs(i - idx)
                                if dist < min_dist:
                                    min_dist = dist
                                    speaker = name

                    # Clean text
                    clean_text = re.sub(r'^[A-Z](?:\s*\([^)]+\))?\s*[:\-]\s*', '', utt)
                    if clean_text == utt:
                        clean_text = re.sub(r'^[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?\s*[:\-]\s*', '', utt)
                    diarized.append({
                        "speaker": speaker,
                        "text": clean_text if clean_text != utt else utt,
                        "confidence": 0.92
                    })

                print(f"✓ Assigned {len(set(name_mapping.keys()))} speakers\n")
                return diarized

        # STRATEGY 2: Clustering-based diarization
        semantic_embeddings = self.embedder.encode(utterances, show_progress_bar=False)

        # Normalize embeddings
        semantic_embeddings = semantic_embeddings / (np.linalg.norm(
            semantic_embeddings, axis=1, keepdims=True) + 1e-8)
        if use_style_features:
            style_features = self._extract_style_features(utterances)
            # Normalize
            style_features = (style_features - style_features.mean(axis=0)) / (
                style_features.std(axis=0) + 1e-8)
            # Combine: 85% semantic, 15% style
            embeddings = np.concatenate([
                semantic_embeddings * 0.85,
                style_features * 0.15
            ], axis=1)
        else:
            embeddings = semantic_embeddings

        # Auto-detect or use specified speaker count
        if num_speakers is None and config.AUTO_DETECT_SPEAKERS:
            num_speakers = self._estimate_speakers(embeddings, config.MAX_SPEAKERS)
            print(f"Auto-detected {num_speakers} speakers")
        elif num_speakers is None:
            num_speakers = min(2, len(utterances))

        # Cluster
        if num_speakers > len(utterances):
            num_speakers = len(utterances)

        clustering = AgglomerativeClustering(n_clusters=num_speakers, linkage='ward')
        labels = clustering.fit_predict(embeddings)

        # Refine with turn-taking if enabled
        if use_turn_taking and len(utterances) > 5:
            labels = self._refine_with_turn_taking(semantic_embeddings, labels)

        # Calculate confidence scores
        confidences = self._calculate_confidences(embeddings, labels)

        # Merge consecutive same-speaker utterances (ONLY for clustering-based approach)
        diarized = []
        current_speaker = None
        current_text = []
        current_confidences = []

        for utt, label, conf in zip(utterances, labels, confidences):
            speaker = f"SPEAKER_{label}"
            if speaker == current_speaker:
                current_text.append(utt)
                current_confidences.append(conf)
            else:
                if current_text:
                    diarized.append({
                        "speaker": current_speaker,
                        "text": " ".join(current_text),
                        "confidence": float(np.mean(current_confidences))
                    })
                current_speaker = speaker
                current_text = [utt]
                current_confidences = [conf]

        if current_text:
            diarized.append({
                "speaker": current_speaker,
                "text": " ".join(current_text),
                "confidence": float(np.mean(current_confidences))
            })

        print(f"Found {len(set(labels))} speakers (merged to {len(diarized)} turns)\n")
        return diarized

Sarcasm detection

In [23]:
class SarcasmDetector:
    def __init__(self):
        print("🔧 Initializing Sarcasm Detector...")
        try:
            self.detector = pipeline(
                "sentiment-analysis",
                model=config.SARCASM_MODEL,
                device=0 if config.DEVICE == 'cuda' else -1
            )
            print("Ready\n")
        except Exception as e:
            print(f"Unavailable: {e}\n")
            self.detector = None

    def detect(self, text):
        if not self.detector or len(text.strip()) < 5:
            return {'is_sarcastic': False, 'score': 0.0}

        try:
            result = self.detector(text[:512])[0]
            label = result['label'].lower()
            score = result['score']
            is_sarcastic = 'negative' in label and score > 0.8
            return {'is_sarcastic': is_sarcastic, 'score': score if is_sarcastic else 0.0}
        except:
            return {'is_sarcastic': False, 'score': 0.0}

GEMINI SUMMARIZER WITH BATCHING & RATE LIMITING

In [24]:
class GeminiSummarizer:
    def __init__(self):
        print("Initializing Gemini...")
        self.model = genai.GenerativeModel(GEMINI_MODEL)
        self.rate_limiter = RateLimiter(delay=config.API_DELAY)
    def _call_with_retry(self, prompt, operation_name="API call"):
        """Make API call with retry logic"""
        for attempt in range(config.RETRY_ATTEMPTS):
            try:
                if attempt > 0:
                    print(f"Retry {attempt}/{config.RETRY_ATTEMPTS} for {operation_name}...")
                    time.sleep(config.RETRY_DELAY)

                response = self.model.generate_content(prompt)
                return response.text.strip()

            except Exception as e:
                error_str = str(e).lower()
                if 'rate limit' in error_str or '429' in error_str:
                    if attempt < config.RETRY_ATTEMPTS - 1:
                        wait_time = config.RETRY_DELAY * (attempt + 1)
                        print(f"Rate limit hit. Waiting {wait_time}s...")
                        time.sleep(wait_time)
                        continue
                    else:
                        print(f"Rate limit exceeded for {operation_name}")
                        return None
                else:
                    print(f"Error in {operation_name}: {e}")
                    if attempt < config.RETRY_ATTEMPTS - 1:
                        continue
                    return None

        return None
    def classify_and_summarize_batch(self, texts, module_groups):
        """
        ULTRA-OPTIMIZED: Classify intents AND summarize modules in ONE API call

        Args:
            texts: List of all dialogue turns
            module_groups: Dict of {module_id: [text1, text2, ...]}

        Returns:
            (intents, module_summaries)
        """
        if not texts:
            return [], {}
        # Build combined prompt
        prompt = "TASK 1: Classify intent for each dialogue turn\n"
        prompt += "TASK 2: Summarize each module\n\n"

        prompt += "="*50 + "\n"
        prompt += "PART A: INTENT CLASSIFICATION\n"
        prompt += "="*50 + "\n"
        prompt += "INTENT TYPES:\n"
        prompt += "- INQUIRY: Asking questions\n"
        prompt += "- COMPLAINT: Expressing problems\n"
        prompt += "- REQUEST: Asking for action\n"
        prompt += "- FEEDBACK: Giving opinions\n"
        prompt += "- STATEMENT: Declarations/explanations\n"
        prompt += "- OTHER: Doesn't fit above\n\n"

        prompt += "DIALOGUE TURNS:\n"
        for i, text in enumerate(texts, 1):
            prompt += f"{i}. {text[:300]}\n"

        prompt += "\n" + "="*50 + "\n"
        prompt += "PART B: MODULE SUMMARIES\n"
        prompt += "="*50 + "\n"
        prompt += "Summarize each module in 2-3 sentences (max 80 words).\n\n"

        for mid, (intent, module_text) in module_groups.items():
            prompt += f"MODULE {mid} ({intent}):\n{module_text[:400]}\n\n"

        prompt += "\n" + "="*50 + "\n"
        prompt += "OUTPUT FORMAT:\n"
        prompt += "="*50 + "\n"
        prompt += "INTENTS:\n"
        prompt += "1|INTENT_TYPE|CONTEXT\n"
        prompt += "2|INTENT_TYPE|CONTEXT\n"
        prompt += "...\n\n"
        prompt += "SUMMARIES:\n"
        prompt += "MODULE 0: [summary]\n"
        prompt += "MODULE 1: [summary]\n"
        prompt += "...\n"

        response = self._call_with_retry(prompt, f"mega-batch ({len(texts)} intents + {len(module_groups)} modules)")

        if not response:
            # Fallback
            intents = [{'primary': 'STATEMENT', 'secondary': 'GENERAL'} for _ in texts]
            summaries = {mid: f"Summary for module {mid}" for mid in module_groups.keys()}
            return intents, summaries

        intents = []
        summaries = {}

        lines = response.strip().split('\n')
        in_intents = False
        in_summaries = False

        for line in lines:
            line = line.strip()

            if 'INTENTS:' in line.upper():
                in_intents = True
                in_summaries = False
                continue
            elif 'SUMMARIES:' in line.upper() or 'MODULE' in line.upper():
                in_intents = False
                in_summaries = True

            if in_intents and '|' in line:
                parts = line.split('|')
                if len(parts) >= 2:
                    intents.append({
                        'primary': parts[1].strip().upper(),
                        'secondary': parts[2].strip() if len(parts) > 2 else 'GENERAL'
                    })

            elif in_summaries and 'MODULE' in line.upper():
                match = re.match(r'MODULE\s+(\d+)\s*[:\-]\s*(.+)', line, re.IGNORECASE)
                if match:
                    mid = int(match.group(1))
                    summary = match.group(2).strip()
                    summaries[mid] = summary

        while len(intents) < len(texts):
            intents.append({'primary': 'STATEMENT', 'secondary': 'GENERAL'})

        for mid in module_groups.keys():
            if mid not in summaries:
                summaries[mid] = f"Summary unavailable for module {mid}"
        return intents[:len(texts)], summaries

    def polish_final_summary(self, summary):
        prompt = f"""Review and polish this summary:

{summary}

Tasks:
1. Check for contradictions or inconsistencies
2. Correct any grammar/spelling errors
3. Return the polished version

OUTPUT FORMAT:
CONSISTENCY: [PASS/FAIL - reason if fail]
POLISHED: [corrected text]
"""

        response = self._call_with_retry(prompt, "polish summary")

        if not response:
            return summary, {'consistent': True, 'issues': None}
        consistency = {'consistent': True, 'issues': None}
        polished = summary

        lines = response.strip().split('\n')
        for i, line in enumerate(lines):
            if 'CONSISTENCY:' in line.upper():
                if 'FAIL' in line.upper():
                    consistency = {'consistent': False, 'issues': line.split(':', 1)[1].strip()}
                else:
                    consistency = {'consistent': True, 'issues': None}
            elif 'POLISHED:' in line.upper():
                polished = '\n'.join(lines[i:]).split(':', 1)[1].strip()
                break

        print(f"Polished (consistent: {consistency['consistent']})\n")
        return polished, consistency
    def summarize_modules_batch(self, module_texts, batch_size=2):
        """Batch summarize modules in groups"""

        if not module_texts:
            return []

        print(f"Batch summarizing {len(module_texts)} modules (batches of {batch_size})...")

        summaries = []
        total_batches = (len(module_texts) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(module_texts), batch_size):
            batch = module_texts[batch_idx:batch_idx + batch_size]
            batch_num = batch_idx // batch_size + 1

            print(f"  Batch {batch_num}/{total_batches} ({len(batch)} modules)...")
            prompt = f"Summarize each module below in exactly 2-3 sentences (max 80 words each).\n\n"
            prompt += f"CRITICAL: Return ONLY the summaries in this exact format:\n"
            prompt += f"1. [summary text]\n"
            prompt += f"2. [summary text]\n"
            prompt += f"etc.\n\n"
            prompt += f"Do NOT include 'Module', 'Summary', or any other labels.\n\n"

            for i, text in enumerate(batch, 1):
                prompt += f"=== TEXT {i} ===\n{text[:500]}\n\n"

            prompt += f"\nNow provide exactly {len(batch)} numbered summaries:"

            if batch_idx > 0:
                self.rate_limiter.wait()

            response = self._call_with_retry(prompt, f"module batch {batch_num}")

            if response:
                batch_summaries = []
                parts = re.split(r'\n\s*(\d+)[\.)]\s*', response)
                for i in range(2, len(parts), 2):
                    if i < len(parts):
                        summary = parts[i].strip()
                        summary = re.sub(r'^(MODULE|Summary|Text)\s*\d*[:\.]?\s*', '', summary, flags=re.IGNORECASE)
                        summary = re.sub(r'\*\*', '', summary)
                        summary = ' '.join(summary.split())

                        if len(summary) > 20:
                            batch_summaries.append(summary)
                if len(batch_summaries) < len(batch):
                    batch_summaries = []
                    for line in response.split('\n'):
                        line = line.strip()
                        if re.match(r'^\d+[\.)]\s+\w', line):
                            summary = re.sub(r'^\d+[\.)]\s*', '', line)
                            summary = re.sub(r'^(MODULE|Summary)\s*\d*[:\.]?\s*', '', summary, flags=re.IGNORECASE)
                            summary = re.sub(r'\*\*', '', summary)
                            if len(summary) > 20:
                                batch_summaries.append(summary)
                while len(batch_summaries) < len(batch):
                    idx = len(batch_summaries)
                    if idx < len(batch):
                        batch_summaries.append(batch[idx][:200] + "...")
                    else:
                        batch_summaries.append("Summary unavailable.")

                summaries.extend(batch_summaries[:len(batch)])
            else:
                summaries.extend([text[:200] + "..." for text in batch])

        print(f"✓ Batch summarized {len(summaries)} modules\n")
        return summaries
    def summarize(self, text, context="dialogue", max_length=200):
        if not text or len(text.strip()) < 10:
            return "Text too short."

        prompt = f"Summarize concisely (max {max_length} words):\n\n{text}\n\nSummary:"
        response = self._call_with_retry(prompt, "summarize")
        return response or "Failed."

    def assemble_final_summary(self, module_summaries):
        """Assemble final summary with rate limiting"""
        if len(module_summaries) == 1:
            return list(module_summaries.values())[0]['abstractive']

        print("🔨 Assembling final summary...")
        self.rate_limiter.wait()  # Delay before assembly

        modules_text = "\n".join([
            f"{i+1}. {s['primary_intent']}: {s['abstractive']}"
            for i, s in enumerate(module_summaries.values())
        ])

        prompt = f"Combine these module summaries into ONE coherent 3-4 sentence summary. Focus on the main issues, actions, and outcomes:\n\n{modules_text}\n\nFinal Summary:"
        response = self._call_with_retry(prompt, "final assembly")
        return response or "Assembly failed."

    def check_consistency(self, summary):
        """Check consistency with rate limiting"""
        print("Checking consistency...")
        self.rate_limiter.wait()  # Delay before consistency check

        prompt = f"Check for contradictions. Reply: CONSISTENT or INCONSISTENT|reason\n\n{summary}\n\nResult:"
        response = self._call_with_retry(prompt, "consistency check")

        if not response:
            return {'consistent': True, 'issues': None}

        if 'CONSISTENT' in response and 'INCONSISTENT' not in response:
            return {'consistent': True, 'issues': None}

        parts = response.split('|', 1)
        return {'consistent': False, 'issues': parts[1] if len(parts) > 1 else "Unknown"}

    def correct_grammar(self, text):
        """Correct grammar (no delay needed, last call)"""
        print("Correcting grammar...")

        prompt = f"Correct grammar (return corrected text only):\n\n{text}\n\nCorrected:"
        response = self._call_with_retry(prompt, "grammar correction")
        return response or text

Intent Graph

In [25]:
class IntentGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.node_id = 0

    def add_utterance(self, text, intent, speaker, sarcasm_info, embedding):
        self.graph.add_node(
            self.node_id,
            text=text,
            intent=intent['primary'],
            speaker=speaker,
            is_sarcastic=sarcasm_info['is_sarcastic'],
            embedding=embedding
        )

        if self.node_id > 0:
            prev_emb = self.graph.nodes[self.node_id - 1]['embedding']
            sim = cosine_similarity([prev_emb], [embedding])[0][0]
            self.graph.add_edge(self.node_id - 1, self.node_id, weight=float(sim))

        self.node_id += 1

    def visualize(self, save_path="intent_graph.png"):
        if len(self.graph.nodes()) == 0:
            return

        try:
            plt.figure(figsize=(12, 8))
            pos = nx.spring_layout(self.graph, k=1.5, iterations=30)

            colors = {'INQUIRY': '#FF6B6B', 'COMPLAINT': '#FF8B94',
                     'REQUEST': '#FFE66D', 'FEEDBACK': '#FCBAD3',
                     'STATEMENT': '#4ECDC4', 'OTHER': '#CCC'}

            node_colors = [colors.get(self.graph.nodes[n]['intent'], '#CCC')
                          for n in self.graph.nodes()]
            node_sizes = [700 if self.graph.nodes[n]['is_sarcastic'] else 400
                         for n in self.graph.nodes()]

            nx.draw_networkx_nodes(self.graph, pos, node_color=node_colors,
                                 node_size=node_sizes, alpha=0.7)
            nx.draw_networkx_edges(self.graph, pos, alpha=0.2, arrows=True)

            labels = {n: f"{n}" for n in self.graph.nodes()}
            nx.draw_networkx_labels(self.graph, pos, labels, font_size=7)

            plt.title("Intent Graph", fontsize=12)
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            print(f"Graph saved: {save_path}")
            plt.close()
        except Exception as e:
            print(f"Visualization failed: {e}")

    def cluster_graph(self):
        if len(self.graph.nodes()) < config.MIN_MODULE_SIZE:
            return {0: list(self.graph.nodes())}

        try:
            G_und = self.graph.to_undirected()
            partition = community_louvain.best_partition(G_und, resolution=1.0)

            modules = defaultdict(list)
            for node, cid in partition.items():
                modules[cid].append(node)

            modules = {k: v for k, v in modules.items()
                      if len(v) >= config.MIN_MODULE_SIZE}

            if not modules:
                modules = {0: list(self.graph.nodes())}

            print(f"{len(modules)} modules\n")
            return modules
        except Exception as e:
            print(f"Clustering failed: {e}, using single module")
            return {0: list(self.graph.nodes())}

Heirarchial Summarizer

In [26]:
class HierarchicalSummarizer:
    def __init__(self, gemini):
        self.gemini = gemini
        self.textrank = TextRankSummarizer()
        self.tokenizer = Tokenizer("english")
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)

    def _extractive_summarize(self, text):
        try:
            parser = PlaintextParser.from_string(text, self.tokenizer)
            summary = self.textrank(parser.document, sentences_count=config.EXTRACTIVE_SENTENCES)
            return " ".join(str(s) for s in summary)
        except:
            sentences = nltk.sent_tokenize(text)
            return " ".join(sentences[:config.EXTRACTIVE_SENTENCES])

    def prepare_module_data(self, graph, modules):
        """Prepare module data for mega-batch processing"""
        module_data = []
        module_groups = {}

        for mid, nodes in modules.items():
            texts = []
            for node in nodes:
                data = graph.nodes[node]
                text = data['text']
                speaker = data['speaker']
                if data['is_sarcastic']:
                    text = f"[SARCASTIC] {text}"
                texts.append(f"[{speaker}] {text}")

            combined = " ".join(texts)
            extractive = self._extractive_summarize(combined)

            intents = [graph.nodes[n]['intent'] for n in nodes]
            primary = Counter(intents).most_common(1)[0][0]

            module_data.append({
                'id': mid,
                'extractive': extractive,
                'primary_intent': primary
            })

            module_groups[mid] = (primary, extractive)

        return module_data, module_groups


Redundancy Remover

In [27]:
class RedundancyRemover:
    def __init__(self):
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)

    def remove_redundancy(self, text):
        sentences = nltk.sent_tokenize(text)
        if len(sentences) <= 1:
            return text

        print(f"Removing redundancy...")
        embeddings = self.embedder.encode(sentences, show_progress_bar=False)

        unique_sent = [sentences[0]]
        unique_emb = [embeddings[0]]

        for i in range(1, len(sentences)):
            sims = cosine_similarity([embeddings[i]], unique_emb)[0]
            if max(sims) < config.REDUNDANCY_THRESHOLD:
                unique_sent.append(sentences[i])
                unique_emb.append(embeddings[i])

        print(f"{len(unique_sent)} unique sentences\n")
        return " ".join(unique_sent)


Main System

In [28]:
class MDSS:
    def __init__(self):

        self.diarizer = SpeakerDiarization()
        self.sarcasm_detector = SarcasmDetector()
        self.gemini = GeminiSummarizer()
        self.redundancy_remover = RedundancyRemover()
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)
        self.whisper = whisper.load_model(config.WHISPER_MODEL)
        print("Whisper loaded\n")
        print("MDSS Ready")
    def transcribe_audio(self, audio_path):
        print(f"Transcribing...")
        result = self.whisper.transcribe(audio_path)
        print(f"Done\n")

        if 'segments' in result:
            return [seg['text'].strip() for seg in result['segments']]
        return nltk.sent_tokenize(result['text'])

    def analyze_dialogue(self, dialogue_input, is_audio=False):
        print("MDSS ANALYSIS :- ")
        start_time = time.time()
        if is_audio:
            utterances = self.transcribe_audio(dialogue_input)
        else:
            lines = [line.strip() for line in dialogue_input.strip().split('\n') if line.strip()]
            if any(re.match(r'^[A-Z](?:\s*\([^)]+\))?\s*[:\-]\s*.+', line) for line in lines):
                utterances = lines
            else:
                utterances = nltk.sent_tokenize(dialogue_input)

        print(f"Processing {len(utterances)} utterances\n")

        print("Step 1: Speaker Diarization")
        diarized = self.diarizer.diarize(utterances)
        print("Step 2: Building Intent Graph")
        temp_graph = IntentGraph()
        for entry in diarized:
            embedding = self.embedder.encode(entry['text'], show_progress_bar=False)
            temp_graph.add_utterance(
                entry['text'],
                {'primary': 'TEMP', 'secondary': 'TEMP'},
                entry['speaker'],
                {'is_sarcastic': False, 'score': 0.0},
                embedding
            )
        print("Step 3: Clustering")
        modules = temp_graph.cluster_graph()
        print("Step 4-5: MEGA-BATCH Processing (Intents + Summaries in 1 call)")
        texts = [entry['text'] for entry in diarized]
        summarizer = HierarchicalSummarizer(self.gemini)
        module_data_temp, module_groups = summarizer.prepare_module_data(temp_graph.graph, modules)
        intents, module_summaries_raw = self.gemini.classify_and_summarize_batch(texts, module_groups)
        print("Step 6: Rebuilding Intent Graph with correct intents")
        intent_graph = IntentGraph()

        for entry, intent in zip(diarized, intents):
            text = entry['text']
            speaker = entry['speaker']
            sarcasm = self.sarcasm_detector.detect(text)
            embedding = self.embedder.encode(text, show_progress_bar=False)
            intent_graph.add_utterance(text, intent, speaker, sarcasm, embedding)

        print(f"{len(intent_graph.graph.nodes())} nodes\n")
        print("Step 6b: Updating module intents...")
        module_summaries = {}
        for mid, nodes in modules.items():
            intents_in_module = [intent_graph.graph.nodes[n]['intent'] for n in nodes]
            primary = Counter(intents_in_module).most_common(1)[0][0]
            temp_data = [m for m in module_data_temp if m['id'] == mid][0]

            module_summaries[mid] = {
                'extractive': temp_data['extractive'],
                'abstractive': module_summaries_raw.get(mid, "Summary unavailable"),
                'primary_intent': primary  # Use CORRECT intent from classified data
            }

        print("\nStep 7: Final Assembly")
        final = self.gemini.assemble_final_summary(module_summaries)
        print("Step 8: Final Polish")
        final, consistency = self.gemini.polish_final_summary(final)
        print("Step 9: Redundancy Removal")
        final = self.redundancy_remover.remove_redundancy(final)
        intent_graph.visualize()

        elapsed = time.time() - start_time
        print(f"COMPLETE - {elapsed:.1f}s")
        print(f"\nStats:")
        print(f"  • Speakers: {len(set([d['speaker'] for d in diarized]))}")
        print(f"  • Modules: {len(modules)}")
        print(f"  • Processing time: {elapsed:.1f}s")
        return {
            'final_summary': final,
            'module_summaries': module_summaries,
            'diarized': diarized,
            'consistency': consistency,
            'num_modules': len(modules),
            'num_speakers': len(set([d['speaker'] for d in diarized])),
            'processing_time': elapsed
        }

Gradio Interface

In [29]:
def create_simple_interface():
    mdss = MDSS()

    def process_text(text):
        if not text or not text.strip():
            return "Please provide text input!"

        try:
            results = mdss.analyze_dialogue(text, is_audio=False)

            out = f"**Processing Time: {results['processing_time']:.1f}s**\n\n"
            out += f"**Summary:**\n{results['final_summary']}\n\n"
            out += f"**Statistics:**\n"
            out += f"-Speakers: {results['num_speakers']}\n"
            out += f"-Modules: {results['num_modules']}\n\n"

            out += "**Module Breakdown:**\n"
            for i, (mid, s) in enumerate(results['module_summaries'].items(), 1):
                out += f"\n**Module {i}** ({s['primary_intent']})\n"
                out += f"→ {s['abstractive']}\n"

            return out
        except Exception as e:
            return f"Error: {str(e)}\n\n{traceback.format_exc()}"

    def process_audio(audio):
        """Audio-only processing"""
        if not audio:
            return "Please record or upload audio!"

        try:
            results = mdss.analyze_dialogue(audio, is_audio=True)

            out = f"**Processing Time: {results['processing_time']:.1f}s**\n\n"
            out += f"**Summary:**\n{results['final_summary']}\n\n"
            out += f"**Statistics:**\n"
            out += f"-Speakers: {results['num_speakers']}\n"
            out += f"-Modules: {results['num_modules']}\n\n"

            out += "**Module Breakdown:**\n"
            for i, (mid, s) in enumerate(results['module_summaries'].items(), 1):
                out += f"\n**Module {i}** ({s['primary_intent']})\n"
                out += f"→ {s['abstractive']}\n"

            return out
        except Exception as e:
            return f"Error: {str(e)}\n\n{traceback.format_exc()}"

    # Create tabbed interface
    with gr.Blocks(theme="soft") as interface:
        gr.Markdown("# MDSS - Multi Dimensional Summarising System")
        gr.Markdown("**Choose your input method:**")

        with gr.Tab("Text Input"):
            text_input = gr.Textbox(
                lines=15,
                placeholder="Paste dialogue here...\n\nExample:\nA: We're behind schedule.\nB: The API is unstable.",
                label="Dialogue Text"
            )
            text_button = gr.Button("Analyze Text", variant="primary")
            text_output = gr.Textbox(label="Results", lines=25)
            text_button.click(process_text, inputs=[text_input], outputs=[text_output])

        with gr.Tab("Audio Input"):
            gr.Markdown("**Note:** Microphone may not work in all Colab environments. Upload audio file if recording fails.")
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Record or Upload Audio",
                format="wav"
            )
            audio_button = gr.Button("Analyze Audio", variant="primary")
            audio_output = gr.Textbox(label="Results", lines=25)
            audio_button.click(process_audio, inputs=[audio_input], outputs=[audio_output])

    return interface


Test Function

In [30]:
def test_mdss():
    print("Testing MDSS\n")
    sample = """A (Project Manager): We're behind schedule on the app launch.
B (Developer): The delay is mainly because the API integration isn't stable.
C (Designer): But the client keeps asking for UI changes.
D (QA Tester): And from my side, the test cases keep failing."""

    try:
        mdss = MDSS()
        results = mdss.analyze_dialogue(sample)
        print("\n" + "="*70)
        print("TEST RESULTS")
        print("="*70)
        print(f"\n**Summary:**\n{results['final_summary']}\n")
        print(f"**Speakers:** {results['num_speakers']} (Expected: 4)")
        print(f"**Modules:** {results['num_modules']}")
        print(f"**Time:** {results['processing_time']:.1f}s")
        print(f"**Consistent:** {results['consistency']['consistent']}")
        print(f"**API Calls:**")
        print("\nTEST PASSED!" if results['num_speakers'] == 4 else "\n SPEAKER COUNT MISMATCH")
        return results
    except Exception as e:
        print(f"Test Failed: {e}")
        traceback.print_exc()
        return None


launching the interface

In [31]:
interface = create_simple_interface()
interface.launch(share=True, debug=True)
print("\nINTERFACE RUNNING...")

Ready

🔧 Initializing Sarcasm Detector...


Device set to use cpu


Ready

🔧 Initializing Gemini...
Whisper loaded

MDSS Ready
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b5dcd4a5d79de48127.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


MDSS ANALYSIS :- 
Processing 4 utterances

Step 1: Speaker Diarization
Extracted 4 speaker names: ['A', 'B', 'C', 'D']
✓ Assigned 4 speakers

Step 2: Building Intent Graph
Step 3: Clustering
2 modules

Step 4-5: MEGA-BATCH Processing (Intents + Summaries in 1 call)
Step 6: Rebuilding Intent Graph with correct intents
4 nodes

Step 6b: Updating module intents...

Step 7: Final Assembly
🔨 Assembling final summary...
Step 8: Final Polish
Polished (consistent: True)

Step 9: Redundancy Removal
Removing redundancy...
3 unique sentences

Graph saved: intent_graph.png
COMPLETE - 24.3s

Stats:
  • Speakers: 4
  • Modules: 2
  • Processing time: 24.3s
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b5dcd4a5d79de48127.gradio.live

INTERFACE RUNNING...
